In [2]:
import pandas as pd
df = pd.read_csv(r"C:\online_retail\data\raw\online_retail_II.csv")

In [3]:
print("Total rows:", len(df))

print("Missing descriptions:", df["Description"].isna().sum())

print("Unique descriptions, excluding missing values:",
      df["Description"].nunique())

print("Unique StockCodes:", df["StockCode"].nunique())

print("\nMost frequent descriptions:")
print(df["Description"].value_counts().head(20))

Total rows: 1067371
Missing descriptions: 4382
Unique descriptions, excluding missing values: 5698
Unique StockCodes: 5305

Most frequent descriptions:
Description
WHITE HANGING HEART T-LIGHT HOLDER    5918
REGENCY CAKESTAND 3 TIER              4412
JUMBO BAG RED RETROSPOT               3469
ASSORTED COLOUR BIRD ORNAMENT         2958
PARTY BUNTING                         2765
STRAWBERRY CERAMIC TRINKET BOX        2613
LUNCH BAG  BLACK SKULL.               2529
JUMBO STORAGE BAG SUKI                2434
HEART OF WICKER SMALL                 2319
JUMBO SHOPPER VINTAGE RED PAISLEY     2297
60 TEATIME FAIRY CAKE CASES           2271
PAPER CHAIN KIT 50'S CHRISTMAS        2215
LUNCH BAG SPACEBOY DESIGN             2206
REX CASH+CARRY JUMBO SHOPPER          2199
HOME BUILDING BLOCK WORD              2192
WOODEN FRAME ANTIQUE WHITE            2189
LUNCH BAG CARS BLUE                   2184
NATURAL SLATE HEART CHALKBOARD        2151
BAKING SET 9 PIECE RETROSPOT          2150
POSTAGE            

In [4]:
# Exclude missing descriptions for this comparison
valid_desc = df.dropna(
    subset=["StockCode", "Description"]
)

# Number of distinct descriptions per StockCode
desc_per_code = (
    valid_desc.groupby("StockCode")["Description"].nunique()
)

print("StockCodes with multiple descriptions:",
      (desc_per_code > 1).sum())

print("\nTop StockCodes with multiple descriptions:")
print(desc_per_code[desc_per_code > 1].sort_values(
    ascending=False
).head(20))


# Number of distinct StockCodes per description
codes_per_desc = (
    valid_desc.groupby("Description")["StockCode"].nunique()
)

print("\nDescriptions associated with multiple StockCodes:",
      (codes_per_desc > 1).sum())

print("\nTop descriptions associated with multiple StockCodes:")
print(codes_per_desc[codes_per_desc > 1].sort_values(
    ascending=False
).head(20))

StockCodes with multiple descriptions: 1232

Top StockCodes with multiple descriptions:
StockCode
20713     9
21181     7
22423     7
22734     7
23084     7
21830     6
22719     6
47566B    6
85175     6
22501     5
22740     5
22502     5
21843     5
23131     5
23343     5
72807A    5
85123A    5
85172     5
22837     4
20665     4
Name: Description, dtype: int64

Descriptions associated with multiple StockCodes: 293

Top descriptions associated with multiple StockCodes:
Description
check                     146
?                          88
damages                    80
damaged                    76
found                      28
missing                    26
sold as set on dotcom      20
Damaged                    17
adjustment                 15
dotcom                     12
Found                       9
Unsaleable, destroyed.      9
thrown away                 9
checked                     8
smashed                     8
??                          7
damages?                    

In [6]:

sc = df['StockCode'].astype(str).str.strip().str.upper()
df['SC_clean'] = sc

# Split into numeric base + letter suffix
parts = sc.str.extract(r'^(?P<base>\d{5})(?P<suffix>[A-Z]*)$')
df['sc_base'] = parts['base']
df['sc_suffix'] = parts['suffix']

# How many codes follow the "5 digits + suffix" pattern?
print("Matches pattern:", parts['base'].notna().mean())
print("Non-matching (POST, M, BANK CHARGES, etc.):")
print(df.loc[parts['base'].isna(), 'SC_clean'].value_counts().head(20))

# Bases that have multiple suffix variants
variants = (df.dropna(subset=['sc_base'])
              .groupby('sc_base')['SC_clean'].nunique())
print("Bases with >1 code:", (variants > 1).sum(), "of", len(variants))

# For those, are descriptions identical or different?
multi = variants[variants > 1].index
sub = df[df['sc_base'].isin(multi)].dropna(subset=['Description'])
g = sub.groupby('sc_base').agg(
    n_codes=('SC_clean', 'nunique'),
    n_desc=('Description', 'nunique'),
    n_prices=('Price', 'nunique'),
)
print("Same description across codes:", (g['n_desc'] == 1).sum())
print("Different descriptions:", (g['n_desc'] > 1).sum())

# Eyeball a few
print(sub[sub['sc_base'] == '85183']
      .groupby(['SC_clean', 'Description'])['Price'].agg(['count', 'median']))

Matches pattern: 0.9942915818398663
Non-matching (POST, M, BANK CHARGES, etc.):
SC_clean
POST            2122
DOT             1446
M               1426
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
GIFT_0001_20      29
GIFT_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
GIFT_0001_10      16
CRUK              16
DCGS0076          15
TEST001           15
DCGS0003          14
Name: count, dtype: int64
Bases with >1 code: 430 of 4198
Same description across codes: 31
Different descriptions: 390
                                              count  median
SC_clean Description                                       
85183A   CHARLIE & LOLA WASTEPAPER BIN BLUE     147    1.25
85183B   CHARLIE & LOLA WASTEPAPER BIN FLORA    178    1.25


In [7]:
# 1. Confirm what the special codes are, using Description
special = df[~df['SC_clean'].str.match(r'^\d{5}[A-Z]*$')]
print(special.groupby('SC_clean')['Description']
      .agg(lambda x: x.dropna().str.upper().mode().iloc[0] if x.notna().any() else None)
      .to_frame('top_description')
      .join(special.groupby('SC_clean').agg(rows=('Price', 'size'),
                                            median_price=('Price', 'median'),
                                            median_qty=('Quantity', 'median'))))

# 2. Flag non-products
non_product = ['POST','DOT','M','C2','D','S','BANK CHARGES','ADJUST','AMAZONFEE',
               'CRUK','TEST001','PADS'] 
df['is_product'] = ~df['SC_clean'].isin(non_product) & ~df['SC_clean'].str.startswith('GIFT_')
print(df['is_product'].value_counts())

# 3. Look at the 31 merge candidates
sub = df[df['sc_base'].isin(multi)].dropna(subset=['Description'])
g = sub.groupby('sc_base').agg(n_codes=('SC_clean','nunique'),
                               n_desc=('Description','nunique'))
same = g[g['n_desc'] == 1].index
print(sub[sub['sc_base'].isin(same)]
      .groupby(['sc_base','SC_clean','Description'])['Price'].agg(['count','median']).head(30))

# 4. Split the 390 into real variants vs casing/whitespace noise
sub = sub.assign(desc_norm=sub['Description'].str.upper().str.strip().str.replace(r'\s+', ' ', regex=True))
g2 = sub.groupby('sc_base')['desc_norm'].nunique()
print("Different even after normalising:", (g2 > 1).sum())

                                  top_description  rows  median_price  \
SC_clean                                                                
ADJUST        ADJUSTMENT BY JOHN ON 26/01/2010 16    67        49.350   
ADJUST2       ADJUSTMENT BY PETER ON JUN 25 2010      3       300.130   
AMAZONFEE                              AMAZON FEE    43      6497.470   
B                                 ADJUST BAD DEBT     6    -24993.965   
BANK CHARGES                         BANK CHARGES   102        42.120   
...                                           ...   ...           ...   
POST                                      POSTAGE  2122        18.000   
S                                         SAMPLES   104        29.995   
SP1002                     KID'S CHALKBOARD/EASEL     3         2.950   
TEST001                   THIS IS A TEST PRODUCT.    15         4.500   
TEST002                   THIS IS A TEST PRODUCT.     2         0.500   

              median_qty  
SC_clean               

for checking does each unique description means an unique product

In [8]:
df['StockCode'].nunique(), df['Description'].nunique()

# StockCodes with more than one description
multi = df.groupby('StockCode')['Description'].nunique()
multi[multi > 1]

# Descriptions shared by more than one StockCode
shared = df.groupby('Description')['StockCode'].nunique()
shared[shared > 1]

Description
3 GARDENIA MORRIS BOXED CANDLES      2
3 WHITE CHOC MORRIS BOXED CANDLES    2
3D DOG PICTURE PLAYING CARDS         2
3D SHEET OF CAT STICKERS             2
3D SHEET OF DOG STICKERS             2
                                    ..
wet                                  2
wet damaged                          5
wet pallet                           3
wet/rusty                            2
wrong code                           2
Name: StockCode, Length: 293, dtype: int64

In [9]:
# StockCodes that have a description somewhere
has_desc = df.dropna(subset=['Description']).groupby('StockCode')['Description'] \
             .agg(lambda s: s.mode().iloc[0])

# Fill missing descriptions from the same StockCode
df['Description'] = df['Description'].fillna(df['StockCode'].map(has_desc))

# What's still missing after the fill
df[df['Description'].isna()]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SC_clean,sc_base,sc_suffix,is_product
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,21646,21646,,True
16186,490736,35983,NaN,-200,2009-12-07 17:13:00,0.0,NaN,United Kingdom,35983,35983,,True
19313,490976,84571,NaN,-120,2009-12-08 16:42:00,0.0,NaN,United Kingdom,84571,84571,,True
20975,491066,35949,NaN,-67,2009-12-09 12:17:00,0.0,NaN,United Kingdom,35949,35949,,True
22854,491167,17013A,NaN,-45,2009-12-10 11:09:00,0.0,NaN,United Kingdom,17013A,17013,A,True
...,...,...,...,...,...,...,...,...,...,...,...,...
600756,542582,DCGS0057,NaN,-6,2011-01-28 15:01:00,0.0,NaN,United Kingdom,DCGS0057,NaN,NaN,True
634150,545566,37477D,NaN,-9,2011-03-03 16:14:00,0.0,NaN,United Kingdom,37477D,37477,D,True
804713,561250,DCGS0071,NaN,-2,2011-07-26 11:51:00,0.0,NaN,United Kingdom,DCGS0071,NaN,NaN,True
804715,561255,DCGS0066P,NaN,-3,2011-07-26 11:52:00,0.0,NaN,United Kingdom,DCGS0066P,NaN,NaN,True


In [11]:
# 1) Confirm it's a clean group
missing = df[df['Description'].isna()]
missing['Price'].eq(0).mean(), missing['Customer ID'].isna().mean()

# 2) Re-check shared descriptions after cleaning codes
chk = df.groupby('Description')['SC_clean'].nunique()
chk[chk > 1]

# 3) Sales-only dataset
sales = df[(df['Price'] > 0) & df['Description'].notna() & df['Customer ID'].notna()]

In [12]:
miss = df[df['Description'].isna()]
(miss['Price'] == 0).mean()                 # share with zero price
miss['Customer ID'].isna().mean()           # share with no customer
(miss['Quantity'] < 0).mean()               # share negative qty

# Any that look like real sales?
miss[(miss['Price'] > 0) & miss['Customer ID'].notna()].shape[0]

0

In [13]:
sales = df[(df['Price'] > 0) & df['Customer ID'].notna()].copy()

sales['StockCode'].nunique()      # real product count
sales['Description'].nunique()    # real description count
sales['Description'].isna().sum() # should now be 0, or close to it

np.int64(0)